In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
#all imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Task 1: Write your code here:
# read the dataset
df = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')

print(f"Dataset shape: {df.shape}")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
#Plot the target distribution (delivery_time)
df['Delivery_Time'].hist(bins=50)

In [ ]:
# Task 1: Write your code here:
dfc= df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
for col in ['Weather', 'Traffic_Level', 'Time_of_Day','Courier_Experience_yrs', 'Delivery_Time' ]:
    dfc[col] = dfc[col].fillna('unknown')

dfc.info()

In [ ]:
# Task 3: Write your code here:
# Do we have duplicate samples?
def check_duplicates(dfc):
  duplicates = dfc.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    dfc.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(dfc)

dfc.duplicated().sum()
dfc.info()

In [ ]:
dfc

In [ ]:
# Task 4: Write your code here:
categorical_cols = dfc.select_dtypes(include= ['object']).columns   #['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
print(categorical_cols)
for col in categorical_cols:
  encoder = LabelEncoder()
  dfc[col] = encoder.fit_transform(dfc[col].astype(str))

dfc

In [ ]:
# Task 5: Write your code here:
feature_cols = dfc.select_dtypes(include=["int64", "float64"]).columns

scaler = StandardScaler()
dfc[feature_cols] = scaler.fit_transform(dfc[feature_cols])
dfc.head()

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
dfc['Delivery_Time'].value_counts()
#kinda imbalanced

In [ ]:
# Task 1: Write your code here:
X = dfc[feature_cols]
y = dfc['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# Task 2,3,4,5: Write your code here:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_train)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")



kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_trian[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
dfc['Delivery_Time'].hist()

In [ ]:
# Task Bonus: Write your code here: